1. Solar Battery Arbitrage/Configuration & Setup:

Imports core libraries (math, pandas, entsoe) and initialises the ENTSO-E client for European energy market data.
Defines countries (AT, BE, BG, CZ, FR, GR, HU, ES) with their full names for reporting.
Models a Tesla Powerwall 3 or similar with 13.5 kWh capacity, 4.6 kW discharge, and 5.0 kW charge rate.
Sets cost assumptions at €7,200 per Powerwall and €1,000 per gateway (ex-VAT, ex-installation).
Arbitrage logic targets covering 20% of average peak-hour energy, using the top-5 price hours per day.
Data range is set to the full calendar year 2024 (UTC), from 1 Jan 2024 to 1 Jan 2025.

In [9]:
import math
import pandas as pd
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError
from requests import HTTPError

# If not already defined:
client = EntsoePandasClient(api_key="7308a46a-2e78-4bf2-a552-3295649e49e3")  # <-- your token

preselected_countries = ['AT','BE','BG','CZ','FR','GR','HU','SI','ES']

country_names = {
    "AT": "Austria",
    "BE": "Belgium",
    "BG": "Bulgaria",
    "CZ": "Czech Republic",
    "FR": "France",
    "GR": "Greece",
    "HU": "Hungary",
    "ES": "Spain",
}

# Powerwall 3 specs
pw_energy_MWh   = 13.5 / 1000.0  # 13.5 kWh
pw_discharge_MW = 4.6  / 1000.0  # 4.6 kW
pw_charge_MW    = 5.0  / 1000.0  # 5.0 kW

# Cost assumptions (excluding VAT, installation)
price_powerwall_eur = 7200.0
price_gateway_eur   = 1000.0  # one gateway per country/system

# Design choices
target_fraction = 0.20        # cover 20% of average peak-hour energy
STRONG_SOLAR_THRESHOLD = 0.20 # strong solar hours = >20% of peak solar
N_PEAK = 5                    # top-5 price hours per day

start_2024 = pd.Timestamp("2024-01-01", tz="UTC")
end_2024   = pd.Timestamp("2025-01-01", tz="UTC")

2. Mean MWh in top‑5 price hours per day (mean_peak_MWh): 

For each country, fetches day-ahead prices (resampled to hourly) and actual generation
from ENTSO-E.
Both series are inner-joined on a shared hourly index; rows with NaNs are dropped.
Days with fewer than 5 valid hours are skipped; the top-5 highest-price hours are selected from the rest.
The mean generation volume across all such peak hours in 2024 is stored in mean_peak_MWh.
Countries with missing data, API errors, or no qualifying days are logged in failed_mean.

In [ ]:
mean_peak_MWh = {}   # country -> mean MWh per peak hour
failed_mean   = []

for cc in preselected_countries:
    print(f"\n=== {cc} ===")
    try:
        # Prices (2024, hourly)
        prices = client.query_day_ahead_prices(country_code=cc,
                                               start=start_2024,
                                               end=end_2024)
        if prices.empty:
            print("No price data")
            failed_mean.append(cc)
            continue
        prices_hourly = prices.resample("1h").mean()

        # Generation: total MW (sum of all Actual Aggregated technologies)
        gen = client.query_generation(cc, start=start_2024, end=end_2024)
        if gen.empty:
            print("No generation data")
            failed_mean.append(cc)
            continue

        if isinstance(gen.columns, pd.MultiIndex):
            mask = gen.columns.get_level_values(1) == "Actual Aggregated"
            gen_agg = gen.loc[:, mask]
        else:
            gen_agg = gen
        total_gen_MW_15min = gen_agg.sum(axis=1)  # MW at 15-min

        # 15-min → hourly MWh: sum MW over 4 intervals * 0.25 h
        total_gen_MWh_hourly = total_gen_MW_15min.resample("1h").sum() * 0.25

        # Align prices and volume
        df = pd.DataFrame(index=prices_hourly.index)
        df["price_EUR_MWh"] = prices_hourly
        df = df.join(total_gen_MWh_hourly.to_frame(name="volume_MWh"), how="inner")
        df = df.dropna()
        if df.empty:
            print("No overlapping price/volume data")
            failed_mean.append(cc)
            continue

        df["date"] = df.index.date

        peak_volumes = []
        for date, day_df in df.groupby("date"):
            if len(day_df) < N_PEAK:
                continue
            top5 = day_df.sort_values("price_EUR_MWh", ascending=False).head(N_PEAK)
            peak_volumes.append(top5["volume_MWh"])

        if not peak_volumes:
            print("No peak hours found")
            failed_mean.append(cc)
            continue

        peak_all = pd.concat(peak_volumes)
        mean_MWh_peak_5 = peak_all.mean()

        mean_peak_MWh[cc] = mean_MWh_peak_5
        print(f"Mean MWh in top-5 price hours (2024): {mean_MWh_peak_5:,.0f} MWh/hour")

    except (NoMatchingDataError, HTTPError, Exception) as e:
        print("Error:", e)
        failed_mean.append(cc)

print("\nCountries where mean peak MWh could not be computed:", failed_mean)


=== AT ===


3. Price Spread: Strong Solar Hours vs. Peak Price Hours (2024)

For each country, fetches day-ahead prices (resampled to hourly) and solar generation
(extracted from the aggregated generation mix, also resampled to hourly), then inner-joins and drops NaNs.
Strong solar hours are identified by hour-of-day whose mean solar output exceeds 20% of the daily maximum.
Separately, the top-5 highest-price hours per day are flagged (days with fewer than 5 observations are skipped).
The mean price is computed for each group, and their difference — the arbitrage price spread — is stored in price_stats.
Countries with missing solar data, no qualifying solar or peak hours, or API errors are logged in failed_price.

In [ ]:
price_stats = {}   # country -> {mean_strong_price, mean_peak_price, price_spread}
failed_price = []

for cc in preselected_countries:
    print(f"\n=== {cc} ===")
    try:
        # Prices
        prices = client.query_day_ahead_prices(cc, start=start_2024, end=end_2024)
        if prices.empty:
            print("No price data")
            failed_price.append(cc)
            continue
        prices_hourly = prices.resample("1h").mean()
        df_p = prices_hourly.to_frame(name="price_EUR_MWh")
        df_p["hour"] = df_p.index.hour
        df_p["date"] = df_p.index.date

        # Generation for Solar
        gen = client.query_generation(cc, start=start_2024, end=end_2024)
        if gen.empty:
            print("No generation data")
            failed_price.append(cc)
            continue

        if isinstance(gen.columns, pd.MultiIndex):
            mask = gen.columns.get_level_values(1) == "Actual Aggregated"
            gen_agg = gen.loc[:, mask].copy()
            gen_agg.columns = gen_agg.columns.get_level_values(0)
        else:
            gen_agg = gen.copy()
        solar = gen_agg.get("Solar")
        if solar is None:
            print("No Solar column")
            failed_price.append(cc)
            continue

        solar_hourly = solar.resample("1h").mean()
        df = df_p.join(solar_hourly.to_frame(name="solar_MW"), how="inner").dropna()
        if df.empty:
            print("No overlap between prices and solar")
            failed_price.append(cc)
            continue

        # Strong solar hours
        hourly_profile = df.groupby("hour")["solar_MW"].mean()
        max_avg = hourly_profile.max()
        if max_avg <= 0:
            print("No meaningful solar")
            failed_price.append(cc)
            continue
        thr_strong = STRONG_SOLAR_THRESHOLD * max_avg
        strong_hours = hourly_profile[hourly_profile > thr_strong].index.tolist()
        if not strong_hours:
            print("No strong solar hours found")
            failed_price.append(cc)
            continue

        df["is_strong"] = df["hour"].isin(strong_hours)

        # Top-5 price hours per day
        peak_flags = []
        for date, day_df in df.groupby("date"):
            if len(day_df) < N_PEAK:
                continue
            top5 = day_df.sort_values("price_EUR_MWh", ascending=False).head(N_PEAK)
            peak_flags.append(pd.Series(True, index=top5.index))
        if not peak_flags:
            print("No peak hours found")
            failed_price.append(cc)
            continue
        peak_flags = pd.concat(peak_flags)
        df["is_peak"] = df.index.isin(peak_flags.index)

        strong_prices = df[df["is_strong"]]["price_EUR_MWh"]
        peak_prices   = df[df["is_peak"]]["price_EUR_MWh"]
        if strong_prices.empty or peak_prices.empty:
            print("Insufficient strong or peak prices")
            failed_price.append(cc)
            continue

        mean_strong = strong_prices.mean()
        mean_peak   = peak_prices.mean()
        spread      = mean_peak - mean_strong

        price_stats[cc] = {
            "mean_strong_price": mean_strong,
            "mean_peak_price":   mean_peak,
            "price_spread":      spread,
        }

        print(f"Mean strong price: {mean_strong:.2f} EUR/MWh, "
              f"mean peak price: {mean_peak:.2f} EUR/MWh, "
              f"spread: {spread:.2f} EUR/MWh")

    except (NoMatchingDataError, HTTPError, Exception) as e:
        print("Error:", e)
        failed_price.append(cc)

print("\nPrice stats available for:", list(price_stats.keys()))
print("Price stats failed for:   ", failed_price)


=== AT ===
Mean strong price: 73.05 EUR/MWh, mean peak price: 119.68 EUR/MWh, spread: 46.63 EUR/MWh

=== BE ===
Mean strong price: 64.24 EUR/MWh, mean peak price: 106.67 EUR/MWh, spread: 42.43 EUR/MWh

=== BG ===
Mean strong price: 85.35 EUR/MWh, mean peak price: 179.11 EUR/MWh, spread: 93.76 EUR/MWh

=== CZ ===
Mean strong price: 74.96 EUR/MWh, mean peak price: 129.71 EUR/MWh, spread: 54.75 EUR/MWh

=== FR ===
Mean strong price: 51.13 EUR/MWh, mean peak price: 89.60 EUR/MWh, spread: 38.47 EUR/MWh

=== GR ===
Mean strong price: 81.51 EUR/MWh, mean peak price: 165.28 EUR/MWh, spread: 83.77 EUR/MWh

=== HU ===
Mean strong price: 87.95 EUR/MWh, mean peak price: 177.35 EUR/MWh, spread: 89.40 EUR/MWh

=== SI ===
Mean strong price: 80.15 EUR/MWh, mean peak price: 145.19 EUR/MWh, spread: 65.04 EUR/MWh

=== ES ===
Mean strong price: 50.42 EUR/MWh, mean peak price: 92.62 EUR/MWh, spread: 42.20 EUR/MWh

Price stats available for: ['AT', 'BE', 'BG', 'CZ', 'FR', 'GR', 'HU', 'SI', 'ES']
Price stat

4. 10‑year ROI (fleet and per unit), with Powerwall constraints: 

For each country with valid data, the fleet is sized to cover 20% of the mean peak-hour grid energy, using whichever is the binding constraint — Powerwall energy capacity (13.5 kWh) or discharge power (4.6 kW).
Deliverable energy is derated by an average 85% utilisation (linearly degrading from 100% to 70% over 10 years).
Annual arbitrage margin is calculated as price_spread × effective_energy_MWh × 365, summed over 10 years.
Total investment covers all Powerwall units plus one gateway (€1,000), and both fleet-level and per-unit ROI are computed.
Results — including sizing, costs, margins, and ROI — are collected into rows_roi for final tabulation.

In [ ]:
pandas as pd

years = 10
days_per_year = 365

# Utilisation: 100% in year 1, 70% in year 10 → average over 10 years ≈ 85%
utilisation_year1 = 1.0
utilisation_year10 = 0.70
avg_utilisation_10y = (utilisation_year1 + utilisation_year10) / 2.0  # 0.85

rows_roi = []

for cc, mean_MWh_peak_5 in mean_peak_MWh.items():
    if cc not in price_stats:
        continue

    spread = price_stats[cc]["price_spread"]  # EUR/MWh

    # Design: fraction of average peak-hour energy to cover
    target_energy_MWh = mean_MWh_peak_5 * target_fraction

    # Powerwall sizing
    n_by_energy = target_energy_MWh / pw_energy_MWh
    n_by_power  = target_energy_MWh / pw_discharge_MW
    n_units     = math.ceil(max(n_by_energy, n_by_power))

    # Fleet limits at nameplate
    fleet_energy_MWh = n_units * pw_energy_MWh
    fleet_power_MWh  = n_units * pw_discharge_MW

    # Maximum deliverable energy in 1 peak hour at nameplate
    nameplate_effective_energy_MWh = min(fleet_energy_MWh, fleet_power_MWh)

    # Average deliverable energy per peak hour over 10 years,
    # accounting for utilisation degrading from 100% to 70%
    effective_energy_MWh = nameplate_effective_energy_MWh * avg_utilisation_10y

    # Costs
    total_battery_cost = n_units * price_powerwall_eur
    total_gateway_cost = price_gateway_eur
    total_investment   = total_battery_cost + total_gateway_cost

    # Revenue: arbitrage on price spread
    annual_margin_EUR = spread * effective_energy_MWh * days_per_year
    total_margin_10y  = annual_margin_EUR * years
    simple_roi_10y = total_margin_10y / total_investment if total_investment > 0 else None

    # Per unit
    inv_per_unit = total_investment / n_units if n_units > 0 else None
    margin_per_unit_10y = total_margin_10y / n_units if n_units > 0 else None
    roi_per_unit_10y = margin_per_unit_10y / inv_per_unit if inv_per_unit else None

    rows_roi.append({
        "Country": country_names.get(cc, cc),
        "Mean peak-hour energy (MWh)": mean_MWh_peak_5,
        "Target energy per peak hour (MWh)": target_energy_MWh,
        "Nameplate energy per peak hour (MWh)": nameplate_effective_energy_MWh,
        "Effective energy per peak hour (MWh)": effective_energy_MWh,
        "Powerwall 3 units": n_units,
        "Price spread (EUR/MWh)": spread,
        "Annual gross margin (EUR)": annual_margin_EUR,
        "10-year gross margin (EUR)": total_margin_10y,
        "Total investment (EUR)": total_investment,
        "Total investment (billion EUR)": total_investment /import math
import pa 1e9,
        "Simple ROI over 10 years (x)": simple_roi_10y,
        "Investment per unit (EUR)": inv_per_unit,
        "10-year gross margin per unit (EUR)": margin_per_unit_10y,
        "ROI per unit over 10 years (x)": roi_per_unit_10y,
    })

roi_df = pd.DataFrame(rows_roi)

summary_roi = (
    roi_df.assign(
        **{
            "Mean peak-hour energy (MWh)":             roi_df["Mean peak-hour energy (MWh)"].round(0),
            "Target energy per peak hour (MWh)":       roi_df["Target energy per peak hour (MWh)"].round(0),
            "Nameplate energy per peak hour (MWh)":    roi_df["Nameplate energy per peak hour (MWh)"].round(0),
            "Effective energy per peak hour (MWh)":    roi_df["Effective energy per peak hour (MWh)"].round(0),
            "Powerwall 3 units":                       roi_df["Powerwall 3 units"].round(0),
            "Price spread (EUR/MWh)":                  roi_df["Price spread (EUR/MWh)"].round(2),
            "Annual gross margin (EUR)":               roi_df["Annual gross margin (EUR)"].round(0),
            "Total investment (billion EUR)":          roi_df["Total investment (billion EUR)"].round(3),
            "10-year gross margin (EUR)":              roi_df["10-year gross margin (EUR)"].round(0),
            "Simple ROI over 10 years (x)":            roi_df["Simple ROI over 10 years (x)"].round(2),
            "Investment per unit (EUR)":               roi_df["Investment per unit (EUR)"].round(0),
            "10-year gross margin per unit (EUR)":     roi_df["10-year gross margin per unit (EUR)"].round(0),
            "ROI per unit over 10 years (x)":          roi_df["ROI per unit over 10 years (x)"].round(2),
        }
    )
    .sort_values("Country")
)

summary_roi[
    [
        "Country",
        "Mean peak-hour energy (MWh)",
        "Target energy per peak hour (MWh)",
        "Nameplate energy per peak hour (MWh)",
        "Effective energy per peak hour (MWh)",
        "Powerwall 3 units",
        "Price spread (EUR/MWh)",
        "Annual gross margin (EUR)",
        "Total investment (billion EUR)",
        "10-year gross margin (EUR)",
        "Simple ROI over 10 years (x)",
        "Investment per unit (EUR)",
        "10-year gross margin per unit (EUR)",
        "ROI per unit over 10 years (x)",
    ]
]

,Country,Mean peak-hour energy (MWh),Target energy per peak hour (MWh),Nameplate energy per peak hour (MWh),Effective energy per peak hour (MWh),Powerwall 3 units,Price spread (EUR/MWh),Annual gross margin (EUR),Total investment (billion EUR),10-year gross margin (EUR),Simple ROI over 10 years (x),Investment per unit (EUR),10-year gross margin per unit (EUR),ROI per unit over 10 years (x)
0,Austria,8498.0,1700.0,1700.0,1445.0,369460,46.63,24588073.0,2.660,245880733.0,0.09,7200.0,666.0,0.09
1,Belgium,2007.0,401.0,401.0,341.0,87245,42.43,5282846.0,0.628,52828457.0,0.08,7200.0,606.0,0.08
2,Bulgaria,1058.0,212.0,212.0,180.0,46014,93.76,6157283.0,0.331,61572827.0,0.19,7200.0,1338.0,0.19
3,Czech Republic,5263.0,1053.0,1053.0,895.0,228847,54.75,17880089.0,1.648,178800895.0,0.11,7200.0,781.0,0.11
4,France,17316.0,3463.0,3463.0,2944.0,752850,38.47,41335777.0,5.421,413357774.0,0.08,7200.0,549.0,0.08
5,Greece,1554.0,311.0,311.0,264.0,67570,83.77,8078350.0,0.487,80783500.0,0.17,7200.0,1196.0,0.17
6,Hungary,3407.0,681.0,681.0,579.0,148124,89.40,18898061.0,1.066,188980608.0,0.18,7200.0,1276.0,0.18
7,Slovenia,509.0,102.0,102.0,87.0,22137,65.04,2054918.0,0.159,20549182.0,0.13,7200.0,928.0,0.13
8,Spain,28602.0,5720.0,5720.0,4862.0,1243583,42.20,74895556.0,8.954,748955556.0,0.08,7200.0,602.0,0.08
